In [ ]:
print("=" * 70)
print("FEB 2025 DATENFEHLER-REPORT")
print("=" * 70)
print()
print("PROBLEM:")
print(f"  • Februar 2025 (01.02–08.02) wurde nur 8 Tage gescraped")
print(f"  • Mentions/Tag: ~200 (statt normal 80–100)")
print(f"  • Anomalie-Faktor: 2,6x höher als Januar-Baseline")
print(f"  • ROOT CAUSE: Incomplete Data Capture (kein Fehler im Scraper)")
print()
print("LÖSUNG:")
print(f"  ✓ Zeitraum einheitlich: 21.04.2022 bis 08.02.2025")
print(f"  ✓ Q1 2025 ist incomplete (8 Tage statt 90)")
print(f"  ✓ Für Vergleiche: tagesgewichtet auf 90-Tage-Basis")
print(f"  ✓ Grafiken: keine Übertreibung durch incomplete Q1")
print()
print("NOTEBOOKS AKTUALISIERT:")
print(f"  ✓ 06_Klima_Begriffe_Analyse.ipynb: END_DATE='2025-02-08'")
print(f"  ✓ 07_optional_Vergleich_Exact_vs_Lemma.ipynb: END_DATE='2025-02-08'")
print(f"  ✓ 08_optional_Suffix_EDA.ipynb: Datenqualitäts-Hinweis hinzugefügt")
print()
print("=" * 70)

## 4. Fehler-Report: Zusammenfassung

In [ ]:
# Quartals-Level Berechnung
df_merged = df_context.merge(
    df_meta[['newspaper_id', 'data_published', 'klima_mentions_count']],
    on='newspaper_id',
    how='left'
)
df_merged['date'] = pd.to_datetime(df_merged['data_published'])
df_merged['quarter'] = df_merged['date'].dt.to_period('Q')

# Filter: SCRAPER_CUTOFF bis END_DATE
df_filtered = df_merged[
    (df_merged['date'] >= SCRAPER_CUTOFF) &
    (df_merged['date'] <= END_DATE)
].copy()

# Quartals-Aggregate
quarterly_stats = df_filtered.groupby('quarter').agg({
    'quarter': 'count',  # Platzhalter für Zeilencount
    'klima_mentions_count': 'sum'
}).rename(columns={'quarter': 'total_rows', 'klima_mentions_count': 'total_mentions'})

# Tag-Counts pro Quartal
daily_counts = df_filtered.groupby('quarter')['date'].nunique()
quarterly_stats['unique_days'] = daily_counts

# Berechne "normalized" (auf 90 Tage hochskaliert) und Raw
quarterly_stats['normalized_mentions'] = (
    quarterly_stats['total_mentions'] / quarterly_stats['unique_days'] * 90
).astype(int)

quarterly_stats['incomplete'] = quarterly_stats['unique_days'] < 90

print("=== QUARTALS-ÜBERSICHT (mit Tages-Normalisierung) ===")
print(quarterly_stats.to_string())
print("\nHinweise:")
print("- Q1 2025: inclusive (8 Tage), incomplete = True")
print("- normalized_mentions: hochskaliert auf 90 Tage")
print("  Nutze diese für zeitliche Vergleiche innerhalb eines vollständigen Jahres")

## 3. Quartals-Analyse mit Tagesgewichtung

In [ ]:
# Diagnostik: newspapers_processed für Feb 2025
conn = sqlite3.connect(db_path)

query_feb = """
SELECT
    data_published,
    COUNT(*) as record_count,
    SUM(klima_mentions_count) as total_mentions,
    ROUND(AVG(klima_mentions_count), 2) as avg_mentions_per_newspaper
FROM newspapers_processed
WHERE data_published >= '2025-02-01' AND data_published <= '2025-02-08'
GROUP BY data_published
ORDER BY data_published
"""

df_feb = pd.read_sql_query(query_feb, conn)
print("=== FEBRUAR 2025 - Tägliche Zusammenfassung ===")
print(df_feb.to_string(index=False))
print(f"\nGesamtmention Feb 2025: {df_feb['total_mentions'].sum()}")
print(f"Durchschnitt pro Tag: {df_feb['total_mentions'].mean():.0f} mentions")

# Vergleich: Januar 2025 Baseline
query_jan = """
SELECT
    data_published,
    SUM(klima_mentions_count) as total_mentions
FROM newspapers_processed
WHERE data_published >= '2025-01-01' AND data_published <= '2025-01-31'
GROUP BY data_published
ORDER BY data_published
"""

df_jan = pd.read_sql_query(query_jan, conn)
print(f"\n=== JANUAR 2025 - Baseline ===")
print(f"Durchschnitt pro Tag: {df_jan['total_mentions'].mean():.0f} mentions (n={len(df_jan)} Tage)")
print(f"Min/Max: {df_jan['total_mentions'].min():.0f} / {df_jan['total_mentions'].max():.0f}")

print(f"\n✓ ANOMALIE-FAKTOR: {df_feb['total_mentions'].mean() / df_jan['total_mentions'].mean():.2f}x höher als Januar")

conn.close()

## 2. Februar 2025 Anomalie-Diagnose

In [ ]:
SCRAPER_CUTOFF = '2022-04-21'
END_DATE = '2025-02-08'
Q1_2025_START = '2025-01-01'
Q1_2025_END = '2025-02-08'

print(f"Zeitraum Analyse: {SCRAPER_CUTOFF} bis {END_DATE}")
print(f"Q1 2025: {Q1_2025_START} bis {Q1_2025_END} (incomplete, 8 Tage statt 90)")

## 1. Einheitliche Zeitperiode-Parameter

**Definition**:
- **SCRAPER_CUTOFF**: 21.04.2022 (Scraper-Update, neue Basis)
- **END_DATE**: 08.02.2025 (letztes gecrawltes Datum)
- **Q1 2025**: Incomplete (8 Tage: 2025-02-01 bis 2025-02-08)

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import os
import sys

# Einheitliche Pfadlogik
notebook_dir = (
    os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
)
sys.path.append(os.path.join(notebook_dir, "..", "pylib"))
from handle_sqlite import read_table_as_dataframe

db_path = os.path.join(notebook_dir, "..", "data_output", "dwh_data.db")

# Daten laden
print("Lade Daten...")
df_context = read_table_as_dataframe('context_processed', db_path)
df_meta = read_table_as_dataframe('newspapers_processed', db_path)

print(f"✓ context_processed: {len(df_context)} Zeilen")
print(f"✓ newspapers_processed: {len(df_meta)} Zeilen")

# Datenqualitäts-Diagnose: Februar 2025 Anomalie

**Ziel**: Diagnose des unerwarteten Anstiegs auf ~1.000 Mentions/Tag in Feb 2025 (statt normal ~80–100)

**Befund**: Feb 2025 (01.02–08.02.2025) wurde nur 8 Tage gescraped. Dies führt zu einem Incomplete-Artefakt bei aggregierten Analysen.

**Lösung**: Einheitliche Regel → Zeitraum: 21.04.2022 bis 08.02.2025 (Scraper-Cutoff bis letztes gecrawltes Datum). Q1 2025 ist incomplete (8 Tage); muss tagesgewichtet werden.

---

⚠️ **Generiert**: 20.03.2026